<a href="https://colab.research.google.com/github/Ahmednoorr/Ahmed-Portfolio-/blob/main/Ahmed_dcs109_final_rcv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
####### RUN THIS CODE BLOCK FIRST #################
########## DO NOT EDIT ############################

# Purpose: A function to help test your code
# Last Edited: Nov 5, 2024

################################################################################
from typing import Callable  # use Callable as a type hint
################################################################################

def printOneTest(_function: Callable, *args: tuple, **kwargs: dict) -> None:
    ''' Function to streamline fruitful-function testing, allowing the user to
        pass in a function, arbitrary number of arguments, and an expected result.

    Parameters:
        - _function: the name of a Callable function (the function name -- not a call)
        - (as part of *args): pass zero or more arguments, each separated by a comma,
            that your function will need when called
        - "expected" (as part of **kwargs): the expected result of the function call
        - [optional (as part of **kwargs)]:
            - "fruitful":  [default True]  if False, student function is expected to print, not return
            - "simple":    [default False] if True, uses "+" and "x" for correct and incorrect
            - "indent":    [default False] if True, prints a tabstop print to each output line

    Example Usage:
        # from printOneTest import *

        def blarb() -> int:
            return 2 + 4 + 6 + 8
        def sumOfInts(a: int, b: int, c: int) -> int:
            return a + b + c
        def printSum(a: int, b: int, c: int) -> None:
            print(f"{a} + {b} + {c} = {a + b + c}")

        def main() -> None:
            printOneTest(blarb, expected = 20)
            printOneTest(sumOfInts, 1, 2, 3, expected = 6)
            printOneTest(sumOfInts, 1, 2, 3, expected = 6, simple = True, indent = True)
            printOneTest(printSum, 1, 2, 3, expected = "1 + 2 + 3 = 6", fruitful = False)

        main()
    '''
    # check for 'indent' keyword argument, defaulting to False if not provided
    try:    _indent = kwargs["indent"]
    except: _indent = False
    _tab = '' if not _indent else '\t'

    # enforce that the first argument is a function reference,
    # not the result of a function call
    if not isinstance(_function, Callable):
        print(f"{_tab}ERROR: first argument to printOneTest must be a function, " + \
              f"not {type(_function).__name__}")
        return

    # grab the frame object associated with the student's calling module, and then import
    # that module (outside of this module) so we can call the function to be tested
    import inspect
    _calling_frame = inspect.stack()[-1][0]
    _student_module = inspect.getmodule(_calling_frame)

    # determine whether the student is passing in a method or function
    # (method will have a .__self__; a function will raise AttributeError)
    try:
        _function.__self__
    except AttributeError as err:

        _is_method = False
    else:
        _is_method = True

    # build the function call string for later evaluating
    _function_call_string = f"{_function.__name__}(" + \
        ','.join(str(repr(_arg)) for _arg in args) + ")"

    # enforce a provided 'expected' keyword argument
    try: _expected = kwargs["expected"]
    except:
        print(f"{_tab}ERROR: must provide 'expected' keyword argument to printOneTest")
        return

    # check for 'fruitful' keyword argument, defaulting to True if not provided
    try:    _is_fruitful = bool(kwargs["fruitful"])
    except: _is_fruitful = True

    try:
        if not _is_fruitful:
            import io
            from contextlib import redirect_stdout
            # student's function uses print rather than return
            _student_stdio =  io.StringIO()
            with redirect_stdout(_student_stdio):
                if _is_method:
                    eval(f"_function.__self__.{_function_call_string}")
                else:
                    # removed _student_module.
                    # eval(f"_student_module.{_function_call_string}")
                    eval(f"{_function_call_string}")
            _result = _student_stdio.getvalue().strip() # result of student's print sans '\n'
        else:
            # fruitful function returns value
            if _is_method:
                _result = eval(f"_function.__self__.{_function_call_string}")
            else:
                # removed _student_module.
                # print(f"_student_module.{_function_call_string}")
                _result = eval(f"{_function_call_string}")
    except Exception as err:
        print(f"{_tab}ERROR in evaluating {_function_call_string}: {err}")
        return

    # enforce that result and expected types match
    if type(_result) != type(_expected):
        print(f"{_tab}ERROR: result type for {_function_call_string} expected to be " + \
              f"{type(_expected).__name__}, not {type(_result).__name__}", end = "")
        if _result is None:
            print(f"\n{_tab}\t --> should you be including a 'fruitful = False' keyword argument?", end = "")
        print()
        return

    _check = "✓"; _ecks = "✗"
    try:    _simple = kwargs["simple"]
    except: pass
    else: (_check, _ecks) = ("+", "x") if _simple else ("✓", "✗")

    if isinstance(_expected, float):
        # for floats, don't use ==; rather, check for "close enough"
        _is_correct = _check if abs(_result - _expected) < 1e-6 else _ecks
    elif isinstance(_expected, str):
        # if the student is using a non-fruitful function to print a floating
        # point result, then == may not work; so try to convert result to a
        # float and if successful, use "close enough" criteria; o/w treat as
        # arbitrary str
        try:     float(_result); float(_expected)
        except:  _is_correct = _check if _result == _expected else _ecks
        else:    _is_correct = _check if abs(float(_result) - float(_expected)) < 1e-6 else _ecks
    else:
        _is_correct = _check if _result == _expected else _ecks

    if _is_method:
        # Consider:  <__main__.main.<locals>.ExampleClass object at 0x1003a7fa0
        # scrap "<__main__.main.<locals>." and just show object class type
        print(f"{_tab}Testing <{repr(_function.__self__).split('.')[-1]}.{_function_call_string}:")
    else:
        print(f"{_tab}Testing {_function_call_string}:")

    ######################
    # UPDATED: 18 Nov 2024
    #print(f"{_tab}[{_is_correct}] Result:   {repr(_result)} (type: {type(_result).__name__})")
    #print(f"{_tab}    Expected: {repr(_expected)} (type: {type(_result).__name__})")
    if isinstance(_result, str) and '\n' in _result:
        print(f"{_tab}[{_is_correct}] Result:")
        lines = _result.split('\n')
        for line in lines: print(f"{_tab}{_tab}{line}")
    else:
        print(f"{_tab}[{_is_correct}] Result:   {repr(_result)} (type: {type(_result).__name__})")

    if isinstance(_expected, str) and '\n' in _expected:
        print(f"{_tab}    Expected:")
        lines = _expected.split('\n')
        for line in lines: print(f"{_tab}{_tab}{line}")
    else:
        print(f"{_tab}    Expected: {repr(_expected)} (type: {type(_result).__name__})")


################# DO NOT EDIT IN ANY WAY #########################


In [ ]:
# Start by pasting your Project 1 Code here
# Purpose: RCV PROJECT
# Name: Ahmed Hassan
# Date Last Modified: 12/13
# Sources Used: Runestone
# Collaborators: Charlie
import random
from typing import List, Optional


# 1. declareWinner function
def declareWinner(name: str, round_number: int, is_random: bool, num_votes_for_winner: int, num_votes_overall: int) -> None:
    """Declare the winner with the candidate's name, the round in which they won,
    whether they were selected by lot, and their vote percentage.
    Parameters:
    name : The name of the candidate who won.
    round_number : The round number in which the candidate won.
    is_random : Whether the winner was selected by lot (True if by lot, False if by majority).
    num_votes_for_winner : The number of votes the winner received.
    num_votes_overall: The total number of votes cast in the election.
    """
    percentage = (num_votes_for_winner / num_votes_overall) * 100

    # Calculate the percentage of votes the winner gets

    #Check if the winner was chosen by random
    if is_random:
        print(f"Candidate {name} wins in round {round_number} by lot with {percentage:.2f}% of the vote.")
    else:
    # Print a message if the winner was selected
        print(f"Candidate {name} wins in round {round_number} with {percentage:.2f}% of the vote.")
        # Print a message if the winner was picked by majority vote.



# 2. findIndexOfCandidate function
def findIndexOfCandidate(candidate_name: str, candidates: list[str]) -> None :
    ''' fucntion to find the index of the candidate with parameteres of candidate_name and candidates
        Parameters:
        candidate_name (str): The name of the candidate whose index is to be found.
        candidates (list[str]): A list of candidate names.'''
    for i in range(len(candidates)):
# Loop through all candidates to find the matching name
        if candidates[i] == candidate_name:
# If the candidate at index i matches the candidate_name
            return i
            #return index or else return none
    return None
#If no candidate matches then return None


# 3. countVotesPerCandidate function
def countVotesPerCandidate(candidates: list[str], kth_choice_votes: list[str]) -> list[int]:
    # counts the total amount of votes that each candidate gets from the list
    vote_counts = [0] * len(candidates)
# Create a list of zeros to count votes for each candidat
    for vote in kth_choice_votes:
# Go through each vote in the list of votes
        i = findIndexOfCandidate(vote, candidates)
        ## Find the candidate that got this vote
        if i is not None:  # If found a match for the candidate in the list.
            vote_counts[i] += 1
             #Add one vote to that candidate's count
    return vote_counts
#Return the list with the total votes for each candidate

# 4. areVoteCountsEqual function
def areVoteCountsEqual(vote_counts: list[int]) -> bool:
    '''The function chekcs to see if the votes are equal
    Parameters:
    vote_counts: a list of vote counts'''
    first = vote_counts[0]
   #Take the first vote count as a starting point for comparison
    for count in vote_counts:
       #Go through each vote count
        if count != first:
           # If any vote count is different from the first one
            return False
    return True
    #then return false beacsue not all of the counts are the same and if they are return true


# 5. sumList function
def sumList(values: list[int]) -> int:

    '''This function gives the sum of the list
    parameters:
    values: a list of intergers to be summed up'''
    total = 0  # Start with a total of zero.
    for v in values:
        total += v
    # Go through each value in the list and then adds current vaules inside the total

    return total
# Return the final total sum of all the values.


# 6. computeSimpleMajorityWinner function

def computeSimpleMajorityWinner(vote_counts: list[int]) -> dict[str, int | None, str, bool]:
    '''computes the simple majority winner based on a simple majority vote.
    Parameters:
    vote_counts: a list of vote counts'''

    # Step 1: Identify the maximum vote count manually
    max_votes = None
    for votes in vote_counts:
        if max_votes is None or votes > max_votes:
            max_votes = votes

    # Step 2: Identify candidates with the maximum vote count
    candidates_with_max_votes = []
    for i in range(len(vote_counts)):
        if vote_counts[i] == max_votes:
            candidates_with_max_votes.append(i)

    # Step 3: Handle different cases based on the number of candidates with the max votes
    if max_votes == 0 or len(candidates_with_max_votes) > 1:
        # If there's a tie or no votes at all, randomly choose a winner
        if len(candidates_with_max_votes) == 1:
            return {"winner": candidates_with_max_votes[0], "by_lot": True}
        return {"winner": None, "by_lot": False}

    # If there is only one winner (no tie)
    return {"winner": candidates_with_max_votes[0], "by_lot": False}

# 7. runElection function
def runElection(round1_candidates: list[str], \
                choice1_votes: list[str], \
                choice2_votes: list[str], \
                choice3_votes: list[str], \
                debug: bool = True) -> None:
    """
    Run a Ranked Choice Voting (RCV) election, printing the candidates and vote counts
    for each round until a winner is determined or only one candidate remains.
    Parameters:
    round1_candidates : A list of candidate names for the first round.
    choice1_votes : A list of candidate names representing the first choice votes.
    choice2_votes : A list of candidate names representing the second choice votes.
    choice3_votes : A list of candidate names representing the third choice votes.
    """
    candidates = copyList(round1_candidates)  # Start with the original list of candidates

    round_number = 1
    while len(candidates) > 1:  # Continue until only one candidate remains or we find a winner
        if debug:
            printSlate(round_number, candidates, choice1_votes, choice2_votes, choice3_votes)

        # Count the votes for this round
        vote_counts = countVotesPerCandidate(candidates, choice1_votes)

        # If no candidate has a majority, check second and third choices
        if None in vote_counts:  # Skip None values that were eliminated candidates
            vote_counts = countVotesPerCandidate(candidates, choice2_votes)

        if None in vote_counts:  # Skip None values that were eliminated candidates
            vote_counts = countVotesPerCandidate(candidates, choice3_votes)

        # Find a simple majority winner or the candidate with the least votes
        winner_result = computeSimpleMajorityWinner(vote_counts)

        if winner_result["winner"] is not None:
            winner_index = winner_result["winner"]
            winner_name = candidates[winner_index]
            declareWinner(winner_name, round_number, winner_result["by_lot"], vote_counts[winner_index], sumList(vote_counts))
            return  # End the function if we have a winner

        # Round 1 didn't yield a winner, so find the candidate with the least votes to eliminate
        min_vote_index = findIndexOfMin(vote_counts)
        eliminated_candidate = candidates[min_vote_index]
        num_votes_eliminated = vote_counts[min_vote_index]
        num_votes_overall = sumList(vote_counts)

        # Declare the candidate eliminated
        declareEliminated(eliminated_candidate, round_number, num_votes_eliminated, num_votes_overall)

        # Remove the eliminated candidate from the list of candidates for Round 2
        candidates = removeCandidate(min_vote_index, candidates)

        # Reassign top-choice votes for those who voted for the eliminated candidate
        reassignTopChoices(eliminated_candidate, round1_candidates, choice1_votes, choice2_votes, choice3_votes)

        # Increment the round number and continue to the next round
        round_number += 1

# 8. declareEliminated function

def declareEliminated(name: str, round_number: int, num_votes: int, num_votes_overall: int) -> None:
    """Declare that a candidate has been eliminated in a particular round and state their percentage of votes.
    parameters:
    name: name of the candidate being eliminated
    round_number: round the candidate is eliminated
    num_votes: num of votes of the eliminated canidate
    num_votes_overall: the total num of votes cast"""

    #calculates the percentage of votes
    percentage = (num_votes / num_votes_overall) * 100
    #f string print declaring candidates votesand elimination
    print(f"Candidate {name} eliminated in round {round_number} with only {percentage:.2f}% of the vote.")

# 9. printslate

def printSlate(round_number: int, candidates: list, choice1v: list, choice2v: list, choice3v: list) -> None:
    """Prints the slate for the current round, including the candidates and vote counts.
    paramters:
    round_number: the current round
    candidates: a list of the candidates
    choice1v: list of votes for 1st choice candidates
    choice2v: list of votes for 2nd choice candidates
    choice3v: list of votes for 3rd choice candidates"""
    print(f"----- ROUND {round_number} -----")
    print(f"Candidates: {candidates}")
    print(f"Choice 1 votes: {choice1v}")
    print(f"Choice 2 votes: {choice2v}")
    print(f"Choice 3 votes: {choice3v}")

# 10. copylist

def copyList(original_list: list) -> list:
    """Creates a new copy of the provided list.
    parameters:
    original list: arbirtary list"""
    #creates empty list to be copied
    copied_list = []
    #fr loop to add for item in og list add the item to the copied list
    for item in original_list:
        copied_list.append(item)
    #and return copied
    return copied_list

# 11. findIndexOfMin

def findIndexOfMin(values: list[int])->int:
    """Returns the index of the minimum value in the list.
    paramters:
    values: a list of ints where the min values index is to be found"""
    #min index is zero
    min_index = 0
    #loop through list from second element
    for i in range(1, len(values)):
        #check if the current value is smaller than the value at the current min index
        if values[i] < values[min_index]:
            #if it is then min index is updates to correct index otherwise return min index
            min_index = i
    return min_index

# 12. removeCandidate

def removeCandidate(index: int, candidates: list[str]) -> list[str]:
    '''This function removes a candidate from the list based on the given index.
    parameters:
    index: index of candidate to be removed from candidate list
    candidates:list of candidates'''
    #creates empty list
    new_list = []
    #loop through candidates list
    for i in range(len(candidates)):
        #if the current index isnt the one to be removed then add the candidate to the new list
        if i != index:
            new_list.append(candidates[i])
    #return the new list with removed candidate
    return new_list


# 13. reassignTopChoices

def reassignTopChoices(eliminated_candidate: str, candidates: list[str], top_choices: list[str], second_choices: list[str], third_choices: list[str]) -> None:
    """Reassigns votes from the eliminated candidate to the next available choices in each voter's list of choices
    Parameters:
    eliminated_candidate : The name of the candidate who has been eliminated
    candidates : A list of remaining candidate names
    top_choices : A list of top choice votes
    second_choices : A list of second choice votes
    third_choices : A list of third choice votes
    """
    # Loop through each top-choice vote
    for i in range(len(top_choices)):
        if top_choices[i] == eliminated_candidate:
            # If the second choice is a valid candidate, reassign the vote
            if second_choices[i] in candidates:
                top_choices[i] = second_choices[i]
            # Else, if the third choice is a valid candidate, reassign the vote
            elif third_choices[i] in candidates:
                top_choices[i] = third_choices[i]
            # Otherwise, set to None (no valid choices left)
            else:
                top_choices[i] = None


In [ ]:





#testing the modified declareWinner function
def main():

    printOneTest(declareWinner, "Garry", 2, False, 7, 10, expected="Candidate Garry wins in round 2 with 70.00% of the vote.", fruitful = False)

    printOneTest(declareWinner, "Zoey", 1, True, 4, 8, expected="Candidate Zoey wins in round 1 by lot with 50.00% of the vote.", fruitful = False)

    printOneTest(declareWinner, "harry", 3, False, 2, 3, expected="Candidate harry wins in round 3 with 66.67% of the vote.", fruitful = False)

    printOneTest(declareWinner, "EjHerrman", 2, False, 1, 2, expected="Candidate EjHerrman wins in round 2 with 50.00% of the vote.", fruitful = False)

#testing the new declareEliminated function

    printOneTest(declareEliminated, "Garry", 3, 2, 9, expected="Candidate Garry eliminated in round 3 with only 22.22% of the vote.", fruitful = False)

    printOneTest(declareEliminated, "Zoey", 2, 3, 10, expected="Candidate Zoey eliminated in round 2 with only 30.00% of the vote.", fruitful = False)

    printOneTest(declareEliminated, "ashk", 1, 1, 4, expected="Candidate ashk eliminated in round 1 with only 25.00% of the vote.", fruitful = False)

    printOneTest(declareEliminated, "oval", 3, 2, 10, expected="Candidate oval eliminated in round 3 with only 20.00% of the vote.", fruitful = False)



 #printslate tests

    candidates = ["Pikachu", "Charmander", "Bulbasaur"]
    choice1v = ["Charmander", "Charmander", "Pikachu", "Charmander", "Bulbasaur"]
    choice2v = ["Pikachu", "Bulbasaur", "Pikachu", "Charmander", "Bulbasaur"]
    choice3v = ["Bulbasaur", "Pikachu", "Pikachu", "Charmander", "Bulbasaur"]

    printSlate(1, candidates, choice1v, choice2v, choice3v)

#copylist tests

    #list of ints
    a_list = [1, 2, 3, 4, 5]
    printOneTest(copyList, a_list, expected=[1, 2, 3, 4, 5], fruitful=True)
    #empty list
    a_list = []
    printOneTest(copyList, a_list, expected=[], fruitful=True)
    #list of strings
    a_list = ["Batman", "Superman", "The Flash"]
    printOneTest(copyList, a_list, expected=["Batman", "Superman", "The Flash"], fruitful=True)

#finIndexofMin tests

    #test in instructions
    printOneTest(findIndexOfMin, [3, 10, 0, 93, 0],  expected=2, fruitful=True)
    #test liist on with one value
    printOneTest(findIndexOfMin, [1],  expected=0, fruitful=True)
    #test list with only same values
    printOneTest(findIndexOfMin, [3, 3, 3],  expected=0, fruitful=True)
    #test list with min at begining
    printOneTest(findIndexOfMin, [1, 10, 11, 93],  expected=0, fruitful=True)
    #test list with min at end
    printOneTest(findIndexOfMin, [3, 10, 33, 93, 0],  expected=4, fruitful=True)


#removeCandidate tests

    #candidate removed begining of list
    candidates = ["Jack", "Jill", "James", "John"]
    printOneTest(removeCandidate, 0, candidates, expected=["Jill", "James", "John"], fruitful=True)
    #candidate removed at end of list
    candidates = ["Ben", "Beaver", "Berry", "Blake"]
    printOneTest(removeCandidate, 3, candidates, expected=["Ben", "Beaver", "Berry"], fruitful=True)
    #candidate removed middle of list
    candidates = ["Horse", "Cow", "Pig"]
    printOneTest(removeCandidate, 1, candidates, expected=["Horse", "Pig"], fruitful=True)
    candidates = ["Ahmed Hassan"]
    printOneTest(removeCandidate, 0, candidates, expected=[], fruitful=True)


#reassignTopChoices tests

    candidates = ["A", "B", "C"]
    top_choices = ["A", "A", "C"]
    second_choices = ["B", "C", "A"]
    third_choices = ["C", "A", "B"]

    #Front of the list
    reassignTopChoices("A", candidates, top_choices, second_choices, third_choices)
    print(top_choices)
    print("expected=['B', 'C', 'C']")

    #Back of the list
    reassignTopChoices("C", candidates, top_choices, second_choices, third_choices)
    print(top_choices)
    print("expected=['B', 'C', A]")

    #Both
    candidates = ["B", "C"]
    top_choices = ["A", "A", "C"]
    second_choices = ["B", "C", "A"]
    third_choices = ["C", "A", "B"]
    reassignTopChoices("A", candidates, top_choices, second_choices, third_choices)
    print(top_choices)
    print("expected=[B, C, 'C']")

#commputesimplemajority tests
    random.seed(12345)
    printOneTest(computeSimpleMajorityWinner, [2, 2, 8], expected={"winner": 2, "by_lot": False})

    printOneTest(computeSimpleMajorityWinner, [0, 0, 0], expected={"winner": None, "by_lot": False})

    printOneTest(computeSimpleMajorityWinner, [2, 2, 0], expected={"winner": None, "by_lot": False})

    printOneTest(computeSimpleMajorityWinner, [1, 2, 3], expected={"winner": 2, "by_lot": False})





def main_rcv_testing():

    # for the tests below, use any four names you like;
    # for "clean" printed output, you may want to make all entries the same
    #   width by including trailing spaces, as in "Alice" below;
    # include exactly four candidates

    names  = ["A", "B", "C", "D"]
    #names = ["Fay", "Jay", "May", "Ray"]
    #names = ["Alice   ", "Frankie ", "Franklin", "Snowball"]

    a,b,c,d = names  # short variables for (max 4) candidates will make list creation below convenient
    x = "x"          # x will indicate a vote that will not be considered in the process


    # <BEGIN> UNCOMMENT THIS BLOCK FOR THE ROUND 1 TESTING PHASE OF runElection

    # candidate B wins 1st round
    candidates = [a,b,c]     # three candidates
    choice1    = [b,b,a,b,b] # b wins in round 1
    choice2    = [x,x,x,x,x]
    choice3    = [x,x,x,x,x]
    runElection(candidates, choice1, choice2, choice3, debug = True)  # use after debugging added
    print("\t Expected: Candidate B wins 1st round w/ 80%")

    print('=' * 70)

    # random candidate wins 1st round
    candidates = [a,b,c]       # use three candidates
    choice1    = [b,b,a,a,c,c] # randomly-selected winner in round 1
    choice2    = [x,x,x,x,x,x]
    choice3    = [x,x,x,x,x,x]
    runElection(candidates, choice1, choice2, choice3, debug = True)  # use after debugging added
    print("\t Expected: Random candidate wins 1st round by lot w/ 33%")

    print('=' * 70)

    # <END> UNCOMMENT THIS BLOCK FOR THE ROUND 1 TESTING PHASE OF runElection



    # <BEGIN> UNCOMMENT THIS BLOCK FOR THE ROUND 2 TESTING PHASE OF runElection

    # candidate A wins 2nd round
    candidates = [a,b,c]      # three candidates
    choice1    = [a,b,c,a,b]  # c will be eliminated in round 1
    choice2    = [b,b,a,b,b]  # choice2[2] replaces choice1[2] --> a wins in round 2
    choice3    = [c,b,b,a,b]
    runElection(candidates, choice1, choice2, choice3, debug = True)  # use after debugging added
    print("\t Expected: Candidate A wins 2nd round w/ 60%")

    print('=' * 70)

    # candidate A or B wins 2nd round
    candidates = [a,b,c]        # three candidates
    choice1    = [a,b,c,a,b,b]  # c will be eliminated in round 1
    choice2    = [x,x,a,x,x,x]  # choice2[2] replaces choice1[2] --> a or b @ random in round 2
    choice3    = [x,x,x,x,x,x]
    runElection(candidates, choice1, choice2, choice3, debug = True)  # use after debugging added
    print("\t Expected: Random candidate A or B wins 2nd round by lot w/ 50%")

    print('=' * 70)

    # <END> UNCOMMENT THIS BLOCK FOR THE ROUND 2 TESTING PHASE OF runElection



    # <BEGIN> UNCOMMENT THIS BLOCK FOR THE ROUND 3 TESTING PHASE OF runElection

    # candidate C wins 3rd round w/ all choice2 replacements
    candidates = [a,b,c,d]            # four candidates
    choice1    = [a,a,b,b,b,c,c,c,d]  # d eleminated in round 1
    choice2    = [c,c,x,x,x,x,x,x,c]  # choice2[-1] replaces choice1[-1] --> still no winner in round 2
    choice3    = [x,x,x,x,x,x,x,x,x]  # choice2[0:2] replaces choice1[0:2] --> c wins round 3
    runElection(candidates, choice1, choice2, choice3, debug = True)  # use after debugging added
    print("\t Expected: Candidate C wins 3rd round w/ 66.67%")

    print('=' * 70)

    # candidate C wins 3rd round w/ replacements from choice2 and choice3
    candidates = [a,b,c,d]            # four candidates
    choice1    = [a,a,b,b,b,c,c,c,d]  # d eleminated in round 1
    choice2    = [d,d,x,x,x,x,x,x,c]  # choice2[-1] replaces choice1[-1] --> still no winner in round 2
    choice3    = [b,c,x,x,x,x,x,x,x]  # choice2[0:2] replaces choice1[0:2] --> c wins round 3
    runElection(candidates, choice1, choice2, choice3, debug = True)  # use after debugging added
    print("\t Expected: Candidate C wins 3rd round w/ 55.56%")

    # <END> UNCOMMENT THIS BLOCK FOR THE ROUND 3 TESTING PHASE OF runElection


    # Provided test cases for runElection

    #winner ound 1 by majoirty
    candidates = ["Pony", "Unicorn", "Pegasus"]
    choice1_votes = ["Pony", "Unicorn", "Pony", "Pony", "Pegasus", "Pony", "Pony"]
    choice2_votes = ["Unicorn", "Pegasus", "Unicorn", "Pegasus", "Unicorn", "Unicorn", "Unicorn"]
    choice3_votes = ["Pegasus", "Pony", "Pegasus", "Unicorn", "Pegasus", "Pegasus", "Pegasus"]
    runElection(candidates, choice1_votes, choice2_votes, choice3_votes, debug=True)

    #winner orund 1 by lot
    candidates = ["Thor", "Batman", "Ironman"]
    choice1_votes = ["Thor", "Batman", "Ironman", "Batman", "Thor", "Ironman", "Batman"]
    choice2_votes = ["Batman", "Ironman", "Thor", "Thor", "Batman", "Thor", "Ironman"]
    choice3_votes = ["Ironman", "Thor", "Batman", "Ironman", "Ironman", "Batman", "Thor"]
    runElection(candidates, choice1_votes, choice2_votes, choice3_votes, debug=True)

    #winner round 2 by majorty
    candidates = ["Me", "You", "Him"]
    choice1_votes = ["Me", "You", "Me", "Me", "Him", "Him", "Me"]
    choice2_votes = ["You", "Him", "You", "Him", "You", "Me", "You"]
    choice3_votes = ["Him", "Me", "Him", "You", "Him", "You", "Him"]
    runElection(candidates, choice1_votes, choice2_votes, choice3_votes, debug=True)

    #winner round 2 by lot
    candidates = ["A", "B", "C"]
    choice1_votes = ["A", "B", "C", "B", "A", "C", "A"]
    choice2_votes = ["B", "C", "A", "A", "B", "A", "B"]
    choice3_votes = ["C", "A", "B", "C", "C", "B", "C"]
    runElection(candidates, choice1_votes, choice2_votes, choice3_votes, debug=True)

    #Winner round 3 by majority

    candidates = ["one", "two", "three"]
    choice1_votes = ["one", "two", "three", "two", "one", "three", "one"]
    choice2_votes = ["two", "three", "one", "one", "two", "one", "two"]
    choice3_votes = ["three", "one", "two", "three", "three", "one", "two"]
    runElection(candidates, choice1_votes, choice2_votes, choice3_votes, debug=True)

    #winner round 3 by lot
    candidates = ["Alice", "Bob", "Charlie"]
    choice1_votes = ["Alice", "Bob", "Charlie", "Bob", "Alice", "Charlie", "Alice"]
    choice2_votes = ["Bob", "Charlie", "Alice", "Alice", "Bob", "Alice", "Bob"]
    choice3_votes = ["Charlie", "Alice", "Bob", "Charlie", "Charlie", "Bob", "Charlie"]
    runElection(candidates, choice1_votes, choice2_votes, choice3_votes, debug=True)


main()              # UNCOMMENT THIS TO USE YOUR TESTS IN YOUR main
main_rcv_testing()   # UNCOMMENT THIS TO USE THE PROVIDED TESTS FOR runElection


Testing declareWinner('Garry',2,False,7,10):
[✓] Result:   'Candidate Garry wins in round 2 with 70.00% of the vote.' (type: str)
    Expected: 'Candidate Garry wins in round 2 with 70.00% of the vote.' (type: str)
Testing declareWinner('Zoey',1,True,4,8):
[✓] Result:   'Candidate Zoey wins in round 1 by lot with 50.00% of the vote.' (type: str)
    Expected: 'Candidate Zoey wins in round 1 by lot with 50.00% of the vote.' (type: str)
Testing declareWinner('harry',3,False,2,3):
[✓] Result:   'Candidate harry wins in round 3 with 66.67% of the vote.' (type: str)
    Expected: 'Candidate harry wins in round 3 with 66.67% of the vote.' (type: str)
Testing declareWinner('EjHerrman',2,False,1,2):
[✓] Result:   'Candidate EjHerrman wins in round 2 with 50.00% of the vote.' (type: str)
    Expected: 'Candidate EjHerrman wins in round 2 with 50.00% of the vote.' (type: str)
Testing declareEliminated('Garry',3,2,9):
[✓] Result:   'Candidate Garry eliminated in round 3 with only 22.22% of the vo